# Model table: Fannie + Freddie

Builds the one table the models train on: one row per loan per month, both agencies, with the features,
the target (did the loan pay off this month?) and a train / validation / test label.
The cleaning steps are the same as in notebook 02.

In [1]:
import sys
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "source"))
import fannie_load_data as fannie
import freddie_load_data as freddie

FANNIE_CACHE = ROOT / "data" / "fannie" / "processed" / "perf_sample_80mb.parquet"
FREDDIE_CACHE = ROOT / "data" / "freddie" / "processed" / "perf_sample_30mb.parquet"
OUT = ROOT / "data" / "combined" / "model_table.parquet"
VINTAGES = (2000, 2025)  # origination years in both datasets

# build the samples on first run only
if not FANNIE_CACHE.exists():
    fannie.load_performance_sample(FANNIE_CACHE, mb_per_file=80)
if not FREDDIE_CACHE.exists():
    freddie.load_performance_sample(FREDDIE_CACHE, mb_per_file=30)

## 1. Put both agencies in the same format

- **30-year loans** originated 2000–2025.
- **Target `prepay`** = 1 if the loan was paid off in full this month (zero-balance code `01`), else 0.
- **`exposure`** = the loan's balance this month, used to turn predictions into CPR (original balance in the first 6 months, since Fannie hides it and Freddie rounds it).
- **Loan purpose:** Freddie's `N` (no-cash-out refi) becomes `R`, as in Fannie. Freddie's `R` ("refi, not specified") becomes missing.

In [2]:
def month_index(col, fmt):
    # date string -> months since year 0 (Freddie uses YYYYMM, Fannie MMYYYY)
    y, m = (pl.col(col).str.slice(0, 4), pl.col(col).str.slice(4, 2)) if fmt == "YYYYMM" else (pl.col(col).str.slice(2, 4), pl.col(col).str.slice(0, 2))
    return y.cast(pl.Int32) * 12 + m.cast(pl.Int32) - 1

COLS = ["agency", "loan_id", "t", "od", "age", "orig_rate", "orig_upb", "ltv", "cltv", "dti", "fico", "num_borrowers",
        "num_units", "first_time_buyer", "loan_purpose", "property_type", "occupancy", "channel", "state", "prepay", "exposure"]
CAT = ["first_time_buyer", "loan_purpose", "property_type", "occupancy", "channel", "state"]

def to_common(lf):
    return (lf.with_columns(od=pl.col("fpd") - 1, age=pl.col("t") - pl.col("fpd") + 1,
                            prepay=(pl.col("zero_bal_code").cast(pl.Utf8) == "01").fill_null(False).cast(pl.Int8))
              .filter(pl.col("orig_term") == 360, pl.col("age") >= 1, (pl.col("od") // 12).is_between(*VINTAGES))
              .with_columns(exposure=pl.when(pl.col("upb_at_removal").is_not_null()).then(pl.col("upb_at_removal"))
                                       .when((pl.col("age") <= 6) | ~(pl.col("cur_upb") > 0)).then(pl.col("orig_upb"))
                                       .otherwise(pl.col("cur_upb")),
                            *[pl.col(c).cast(pl.Utf8) for c in CAT],
                            *[pl.col(c).cast(pl.Int32) for c in ["ltv", "cltv", "dti", "fico", "num_borrowers", "num_units"]])
              .select(COLS))

fannie_lf = pl.scan_parquet(FANNIE_CACHE).with_columns(
    agency=pl.lit("fannie"), t=month_index("rep_period", "MMYYYY"), fpd=month_index("first_pay_date", "MMYYYY"),
    fico=pl.coalesce("fico", "fico_classic_orig"))
freddie_lf = pl.scan_parquet(FREDDIE_CACHE).with_columns(
    agency=pl.lit("freddie"), t=month_index("rep_period", "YYYYMM"), fpd=month_index("first_pay_date", "YYYYMM"),
    loan_purpose=pl.col("loan_purpose").cast(pl.Utf8).replace({"R": None, "N": "R"}))

panel = pl.concat([to_common(fannie_lf), to_common(freddie_lf)]).collect()
panel.shape

(40037594, 21)

## 2. Market rate and refi incentive

- **`mkt_rate`** = average rate on new 30-year loans in our data last month (both agencies), a stand-in for the mortgage rate a borrower could get today.
- **`incentive`** = the loan's rate minus `mkt_rate`. Positive means the borrower could save by refinancing.

In [3]:
new_loans = panel.group_by("agency", "loan_id").agg(pl.col("od", "orig_rate", "orig_upb").first())
mkt = (new_loans.group_by("od")
       .agg(mkt_rate=(pl.col("orig_rate") * pl.col("orig_upb")).sum() / pl.col("orig_upb").sum(), n=pl.len())
       .filter(pl.col("n") >= 30))
all_months = pl.DataFrame({"od": np.arange(mkt["od"].min(), panel["t"].max() + 1, dtype=np.int32)})
mkt = all_months.join(mkt, on="od", how="left").with_columns(pl.col("mkt_rate").interpolate().forward_fill())

panel = (panel.join(mkt.select(t=pl.col("od") + 1, mkt_rate="mkt_rate"), on="t", how="left")  # last month's rate
              .with_columns(incentive=pl.col("orig_rate") - pl.col("mkt_rate")))

## 3. Time split

The model is tested on years it has never seen, as it would be in real use:

| Split | Reporting years | Used for |
|---|---|---|
| train | 2000–2018 | fitting the model |
| valid | 2019–2021 | choosing settings, early stopping |
| test | 2022 onward | final score, used once |

In [4]:
panel = panel.with_columns(
    year=pl.col("t") // 12,
    month=(pl.col("t") % 12 + 1).cast(pl.Int8),  # calendar month, for seasonality (summer moves)
).with_columns(
    split=pl.when(pl.col("year") <= 2018).then(pl.lit("train"))
            .when(pl.col("year") <= 2021).then(pl.lit("valid"))
            .otherwise(pl.lit("test")))

panel = panel.filter(pl.col("incentive").is_not_null()).with_columns(
    [pl.col(c).cast(pl.Categorical) for c in CAT + ["agency", "split"]])

(panel.group_by("split", "agency")
      .agg(loans=pl.col("loan_id").n_unique(), rows=pl.len(), payoffs=pl.col("prepay").sum(),
           payoff_pct=100 * pl.col("prepay").mean())
      .sort("split", "agency"))

split,agency,loans,rows,payoffs,payoff_pct
cat,cat,u32,u32,i64,f64
"""test""","""fannie""",280023,6510302,46266,0.710658
"""test""","""freddie""",242022,6579181,45849,0.69688
"""train""","""fannie""",279890,11745447,187920,1.599939
"""train""","""freddie""",222311,9186578,141003,1.534881
"""valid""","""fannie""",145951,3237555,72283,2.232642
"""valid""","""freddie""",122304,2778531,57539,2.070842


## 4. Save

Features the model will use: `incentive`, `mkt_rate`, `age`, `orig_rate`, `orig_upb`, `fico`, `ltv`, `cltv`, `dti`,
`num_borrowers`, `num_units`, `month`, `first_time_buyer`, `loan_purpose`, `property_type`, `occupancy`, `channel`, `state`, `agency`.
`loan_id`, `t`, `year`, `exposure` and `split` are kept for evaluation, not as features.

In [5]:
OUT.parent.mkdir(parents=True, exist_ok=True)
panel.write_parquet(OUT)
print(f"{OUT.relative_to(ROOT)}: {panel.height:,} rows, {panel.width} columns, "
      f"{OUT.stat().st_size / 1e6:.0f} MB on disk, {panel.estimated_size() / 1e9:.1f} GB in memory")
panel.schema

data/combined/model_table.parquet: 40,037,594 rows, 26 columns, 207 MB on disk, 5.1 GB in memory


Schema([('agency', Categorical),
        ('loan_id', String),
        ('t', Int32),
        ('od', Int32),
        ('age', Int32),
        ('orig_rate', Float64),
        ('orig_upb', Float64),
        ('ltv', Int32),
        ('cltv', Int32),
        ('dti', Int32),
        ('fico', Int32),
        ('num_borrowers', Int32),
        ('num_units', Int32),
        ('first_time_buyer', Categorical),
        ('loan_purpose', Categorical),
        ('property_type', Categorical),
        ('occupancy', Categorical),
        ('channel', Categorical),
        ('state', Categorical),
        ('prepay', Int8),
        ('exposure', Float64),
        ('mkt_rate', Float64),
        ('incentive', Float64),
        ('year', Int32),
        ('month', Int8),
        ('split', Categorical)])